# 01 — Baseline / 01 Baseline

**Chapter 15 — File 1 of 6 / 第15章 — 第1个文件（共6个）**

---

## Summary / 总结

This script demonstrates **Import necessary libraries for preprocessing and modeling**.

本脚本演示 **Import necessary libraries for preprocessing and modeling**。

---
## Step 1 — Import necessary libraries for preprocessing and modeling

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, FunctionTransformer
from sklearn.ensemble import \
    GradientBoostingRegressor, BaggingRegressor, RandomForestRegressor

---
## Step 2 — Load the dataset

In [ ]:
Ames = pd.read_csv("Ames.csv")

---
## Step 3 — Adjust data types for categorical variables

In [ ]:
for col in ["MSSubClass", "YrSold", "MoSold"]:
    Ames[col] = Ames[col].astype("object")

---
## Step 4 — Exclude "PID" and "SalePrice" from features and handle the "Electrical" column

In [ ]:
numeric_features = Ames.select_dtypes(include=["int64", "float64"]) \
                       .drop(columns=["PID", "SalePrice"]).columns
categorical_features = Ames.select_dtypes(include=["object"]).columns \
                           .difference(["Electrical"])
electrical_feature = ["Electrical"]

---
## Step 5 — Manually specify the categories for ordinal encoding according to the data dictionary

In [ ]:
ordinal_order = {

---
## Step 6 — Electrical system

In [ ]:
"Electrical": ["Mix", "FuseP", "FuseF", "FuseA", "SBrkr"],

---
## Step 7 — General shape of property

In [ ]:
"LotShape": ["IR3", "IR2", "IR1", "Reg"],

---
## Step 8 — Type of utilities available

In [ ]:
"Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],

---
## Step 9 — Slope of property

In [ ]:
"LandSlope": ["Sev", "Mod", "Gtl"],

---
## Step 10 — Evaluates the quality of the material on the exterior

In [ ]:
"ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 11 — Evaluates the present condition of the material on the exterior

In [ ]:
"ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 12 — Height of the basement

In [ ]:
"BsmtQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 13 — General condition of the basement

In [ ]:
"BsmtCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 14 — Walkout or garden level basement walls

In [ ]:
"BsmtExposure": ["None", "No", "Mn", "Av", "Gd"],

---
## Step 15 — Quality of basement finished area

In [ ]:
"BsmtFinType1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],

---
## Step 16 — Quality of second basement finished area

In [ ]:
"BsmtFinType2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],

---
## Step 17 — Heating quality and condition

In [ ]:
"HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 18 — Kitchen quality

In [ ]:
"KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 19 — Home functionality

In [ ]:
"Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],

---
## Step 20 — Fireplace quality

In [ ]:
"FireplaceQu": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 21 — Interior finish of the garage

In [ ]:
"GarageFinish": ["None", "Unf", "RFn", "Fin"],

---
## Step 22 — Garage quality

In [ ]:
"GarageQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 23 — Garage condition

In [ ]:
"GarageCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 24 — Paved driveway

In [ ]:
"PavedDrive": ["N", "P", "Y"],

---
## Step 25 — Pool quality

In [ ]:
"PoolQC": ["None", "Fa", "TA", "Gd", "Ex"],

---
## Step 26 — Fence quality

In [ ]:
"Fence": ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"]
}

---
## Step 27 — Extract list of ALL ordinal features from dictionary

In [ ]:
ordinal_features = list(ordinal_order.keys())

---
## Step 28 — List of ordinal features except Electrical

In [ ]:
ordinal_except_electrical = [feat for feat in ordinal_features if feat != "Electrical"]

---
## Step 29 — Define transformations for various feature types

In [ ]:
electrical_transformer = Pipeline(steps=[
    ("impute_electrical", SimpleImputer(strategy="most_frequent")),
    ("ordinal_electrical", OrdinalEncoder(categories=[ordinal_order["Electrical"]]))
])

numeric_transformer = Pipeline(steps=[
    ("impute_mean", SimpleImputer(strategy="mean"))
])

---
## Step 30 — Updated categorical imputer using SimpleImputer

In [ ]:
categorical_imputer = SimpleImputer(strategy="constant", fill_value="None")

ordinal_transformer = Pipeline([
    ("impute_ordinal", categorical_imputer),
    ("ordinal", OrdinalEncoder(categories=[ordinal_order[feat]
                                           for feat in ordinal_features
                                           if feat in ordinal_except_electrical]))
])

nominal_features = [feat for feat in categorical_features if feat not in ordinal_features]
categorical_transformer = Pipeline([
    ("impute_nominal", categorical_imputer),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

---
## Step 31 — Combined preprocessor for numeric, ordinal, nominal, and specific electrical data

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("electrical", electrical_transformer, ["Electrical"]),
        ("num", numeric_transformer, numeric_features),
        ("ordinal", ordinal_transformer, ordinal_except_electrical),
        ("nominal", categorical_transformer, nominal_features)
])

---
## Step 32 — Define model pipelines including Gradient Boosting Regressor

In [ ]:
models = {
    "Decision Tree (1 Tree)": DecisionTreeRegressor(random_state=42),
    "Bagging Regressor (100 Decision Trees)":
        BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),
                         n_estimators=100, random_state=42),
    "Bagging Regressor (200 Decision Trees)":
        BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),
                         n_estimators=200, random_state=42),
    "Random Forest (Default of 100 Trees)":
        RandomForestRegressor(random_state=42),
    "Random Forest (200 Trees)":
        RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting Regressor (Default of 100 Trees)":
        GradientBoostingRegressor(random_state=42),
    "Gradient Boosting Regressor (200 Trees)":
        GradientBoostingRegressor(n_estimators=200, random_state=42)
}

---
## Step 33 — Evaluate models using cross-validation and print results

In [ ]:
results = {}
for name, model in models.items():
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", model)
    ])
    scores = cross_val_score(model_pipeline,
                             Ames.drop(columns="SalePrice"), Ames["SalePrice"], cv=5)
    results[name] = round(scores.mean(), 4)
    print(f"{name}: Mean CV R^2 = {results[name]}")

---
## Learning Notes / 学习笔记

- **概念**: Import necessary libraries for preprocessing and modeling 是机器学习中的常用技术。  
  *Import necessary libraries for preprocessing and modeling is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Baseline / 01 Baseline
# Complete Code / 完整代码
# ===============================

# Import necessary libraries for preprocessing and modeling
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, FunctionTransformer
from sklearn.ensemble import \
    GradientBoostingRegressor, BaggingRegressor, RandomForestRegressor

# Load the dataset
Ames = pd.read_csv("Ames.csv")

# Adjust data types for categorical variables
for col in ["MSSubClass", "YrSold", "MoSold"]:
    Ames[col] = Ames[col].astype("object")

# Exclude "PID" and "SalePrice" from features and handle the "Electrical" column
numeric_features = Ames.select_dtypes(include=["int64", "float64"]) \
                       .drop(columns=["PID", "SalePrice"]).columns
categorical_features = Ames.select_dtypes(include=["object"]).columns \
                           .difference(["Electrical"])
electrical_feature = ["Electrical"]

# Manually specify the categories for ordinal encoding according to the data dictionary
ordinal_order = {
    # Electrical system
    "Electrical": ["Mix", "FuseP", "FuseF", "FuseA", "SBrkr"],
    # General shape of property
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    # Type of utilities available
    "Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],
    # Slope of property
    "LandSlope": ["Sev", "Mod", "Gtl"],
    # Evaluates the quality of the material on the exterior
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    # Evaluates the present condition of the material on the exterior
    "ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    # Height of the basement
    "BsmtQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # General condition of the basement
    "BsmtCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # Walkout or garden level basement walls
    "BsmtExposure": ["None", "No", "Mn", "Av", "Gd"],
    # Quality of basement finished area
    "BsmtFinType1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    # Quality of second basement finished area
    "BsmtFinType2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    # Heating quality and condition
    "HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],
    # Kitchen quality
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    # Home functionality
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    # Fireplace quality
    "FireplaceQu": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # Interior finish of the garage
    "GarageFinish": ["None", "Unf", "RFn", "Fin"],
    # Garage quality
    "GarageQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # Garage condition
    "GarageCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # Paved driveway
    "PavedDrive": ["N", "P", "Y"],
    # Pool quality
    "PoolQC": ["None", "Fa", "TA", "Gd", "Ex"],
    # Fence quality
    "Fence": ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"]
}

# Extract list of ALL ordinal features from dictionary
ordinal_features = list(ordinal_order.keys())

# List of ordinal features except Electrical
ordinal_except_electrical = [feat for feat in ordinal_features if feat != "Electrical"]

# Define transformations for various feature types
electrical_transformer = Pipeline(steps=[
    ("impute_electrical", SimpleImputer(strategy="most_frequent")),
    ("ordinal_electrical", OrdinalEncoder(categories=[ordinal_order["Electrical"]]))
])

numeric_transformer = Pipeline(steps=[
    ("impute_mean", SimpleImputer(strategy="mean"))
])

# Updated categorical imputer using SimpleImputer
categorical_imputer = SimpleImputer(strategy="constant", fill_value="None")

ordinal_transformer = Pipeline([
    ("impute_ordinal", categorical_imputer),
    ("ordinal", OrdinalEncoder(categories=[ordinal_order[feat]
                                           for feat in ordinal_features
                                           if feat in ordinal_except_electrical]))
])

nominal_features = [feat for feat in categorical_features if feat not in ordinal_features]
categorical_transformer = Pipeline([
    ("impute_nominal", categorical_imputer),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combined preprocessor for numeric, ordinal, nominal, and specific electrical data
preprocessor = ColumnTransformer(
    transformers=[
        ("electrical", electrical_transformer, ["Electrical"]),
        ("num", numeric_transformer, numeric_features),
        ("ordinal", ordinal_transformer, ordinal_except_electrical),
        ("nominal", categorical_transformer, nominal_features)
])

# Define model pipelines including Gradient Boosting Regressor
models = {
    "Decision Tree (1 Tree)": DecisionTreeRegressor(random_state=42),
    "Bagging Regressor (100 Decision Trees)":
        BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),
                         n_estimators=100, random_state=42),
    "Bagging Regressor (200 Decision Trees)":
        BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),
                         n_estimators=200, random_state=42),
    "Random Forest (Default of 100 Trees)":
        RandomForestRegressor(random_state=42),
    "Random Forest (200 Trees)":
        RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting Regressor (Default of 100 Trees)":
        GradientBoostingRegressor(random_state=42),
    "Gradient Boosting Regressor (200 Trees)":
        GradientBoostingRegressor(n_estimators=200, random_state=42)
}

# Evaluate models using cross-validation and print results
results = {}
for name, model in models.items():
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", model)
    ])
    scores = cross_val_score(model_pipeline,
                             Ames.drop(columns="SalePrice"), Ames["SalePrice"], cv=5)
    results[name] = round(scores.mean(), 4)
    print(f"{name}: Mean CV R^2 = {results[name]}")

---

➡️ **Next / 下一步**: File 2 of 6